# P2b paragraph localization — pilot
Original VI query only. No labels/local evaluation. Exact proposed spans are scored without
truncation; oversize inputs are marked `too_long`, never scored on a hidden prefix.
Pinned BGE revision; two independent GPU workers. Optional Kaggle secret `HF_TOKEN`.
Download only `p2b_pilot_scores.zip`. Pilot is a structural/runtime check, not a quality gate.

In [ ]:
%pip install -q 'transformers>=4.51,<5' 'accelerate>=1.6,<2' 'huggingface-hub>=0.30,<1' 'pyarrow>=18,<22' 'pandas>=2.2,<3'

In [ ]:
import os, json, hashlib, subprocess, sys, time, zipfile
from pathlib import Path
import pandas as pd
import torch
SCOPE='pilot'
WORK=Path('/kaggle/temp/p2b_'+SCOPE); WORK.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']='/kaggle/temp/p2b_hf_cache'
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret('HF_TOKEN')
    if token: os.environ['HF_TOKEN']=token
except Exception:
    pass
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while block:=f.read(4*1024*1024): h.update(block)
    return h.hexdigest()
assert torch.cuda.device_count()==2
paths=list(Path('/kaggle/input').rglob('p2b_manifest.json'))
assert len(paths)==1,paths
manifest=json.loads(paths[0].read_text())
pair_path=paths[0].parent/'p2b_pairs.parquet'
assert manifest['scope']==SCOPE and manifest['schema']=='p2b_localization_v1'
assert sha(pair_path)==manifest['pair_sha256']
print(manifest,flush=True)


In [ ]:
WORKER='"""Pinned BGE exact-input scorer. Run as one subprocess per Kaggle GPU."""\nimport argparse\nimport hashlib\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport transformers\nfrom transformers import AutoTokenizer, AutoModelForSequenceClassification\n\nMODEL = \'BAAI/bge-reranker-v2-m3\'\nREVISION = \'953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e\'\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--input\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--shard\', type=int, choices=[0, 1], required=True)\n    parser.add_argument(\'--batch-size\', type=int, default=8)\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    sha = hashlib.sha256()\n    with args.input.open(\'rb\') as f:\n        while block := f.read(4*1024*1024):\n            sha.update(block)\n    identity = dict(input_sha256=sha.hexdigest(), model=MODEL, revision=REVISION,\n                    shard=args.shard, max_tokens=1024, batch_size=args.batch_size,\n                    torch_version=torch.__version__, transformers_version=transformers.__version__,\n                    worker_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(), truncation=False)\n    config = args.output/\'identity.json\'\n    if config.exists() and json.loads(config.read_text()) != identity:\n        raise RuntimeError(\'Refusing stale checkpoints\')\n    config.write_text(json.dumps(identity))\n    frame = pd.read_parquet(args.input)\n    owned = set(sorted(frame.query_id.unique())[args.shard::2])\n    frame = frame[frame.query_id.isin(owned)]\n    paths = sorted(args.output.glob(\'part_*.parquet\'))\n    done = set()\n    for path in paths:\n        ids = pd.read_parquet(path, columns=[\'pair_id\']).pair_id.tolist()\n        if done.intersection(ids) or len(ids) != len(set(ids)):\n            raise RuntimeError(\'Duplicate checkpoint IDs\')\n        done.update(ids)\n    if not done <= set(frame.pair_id):\n        raise RuntimeError(\'Unknown checkpoint IDs\')\n    pending = frame[~frame.pair_id.isin(done)]\n    tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)\n    model = AutoModelForSequenceClassification.from_pretrained(\n        MODEL, revision=REVISION, torch_dtype=torch.float16).to(\'cuda:0\').eval()\n    if model.config._commit_hash != REVISION:\n        raise RuntimeError(\'Unexpected model revision\')\n    records = []\n    part = len(paths)\n    start_time = time.monotonic()\n    for start in range(0, len(pending), args.batch_size):\n        batch = pending.iloc[start:start+args.batch_size]\n        encoded = tokenizer(batch.query_text.tolist(), batch.passage_text.tolist(),\n                            truncation=False, padding=False)\n        lengths = [len(ids) for ids in encoded[\'input_ids\']]\n        eligible = [i for i, length in enumerate(lengths) if length <= 1024]\n        logits = {}\n        if eligible:\n            selected = [{key: values[i] for key, values in encoded.items()} for i in eligible]\n            padded = tokenizer.pad(selected, padding=True, return_tensors=\'pt\').to(\'cuda:0\')\n            with torch.inference_mode():\n                raw = model(**padded).logits.float().reshape(-1).cpu().numpy()\n            if not np.isfinite(raw).all():\n                raise RuntimeError(\'Nonfinite model score\')\n            logits = dict(zip(eligible, raw.tolist()))\n        for i, row in enumerate(batch.itertuples(index=False)):\n            records.append(dict(pair_id=row.pair_id, status=\'ok\' if i in logits else \'too_long\',\n                                input_tokens=lengths[i], raw_score=logits.get(i, float(\'nan\')),\n                                revision=REVISION, shard=args.shard))\n        if len(records) >= 128 or start+args.batch_size >= len(pending):\n            target = args.output/f\'part_{part:06d}.parquet\'\n            temporary = target.with_suffix(\'.tmp\')\n            pd.DataFrame(records).to_parquet(temporary, index=False)\n            os.replace(temporary, target)\n            records = []\n            part += 1\n            completed = min(start+args.batch_size, len(pending))\n            print(json.dumps(dict(done=len(done)+completed, total=len(frame),\n                                  pairs_s=round(completed/max(time.monotonic()-start_time, .01), 2))), flush=True)\n    scores = pd.concat([pd.read_parquet(p) for p in sorted(args.output.glob(\'part_*.parquet\'))])\n    if scores.pair_id.duplicated().any() or set(scores.pair_id) != set(frame.pair_id):\n        raise RuntimeError(\'Score coverage failure\')\n    scores.to_parquet(args.output/\'scores.parquet\', index=False)\n\n\nif __name__ == \'__main__\':\n    main()\n'
worker_path=WORK/'worker.py'
worker_path.write_text(WORKER,encoding='utf-8')

In [ ]:
processes=[]; logs=[]
for gpu in range(2):
    log_path=WORK/f'gpu{gpu}.log'; log=log_path.open('w'); logs.append((log,log_path))
    env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(gpu)
    cmd=[sys.executable,str(worker_path),'--input',str(pair_path),'--output',str(WORK/f'gpu{gpu}'),'--shard',str(gpu)]
    processes.append(subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,env=env))
while any(p.poll() is None for p in processes):
    time.sleep(30)
    print({'returncodes':[p.poll() for p in processes],
        'tail':[path.read_text(errors='replace').splitlines()[-1:] for _,path in logs]},flush=True)
for log,_ in logs: log.close()
for p,(_,path) in zip(processes,logs):
    if p.returncode: raise RuntimeError(path.read_text(errors='replace')[-5000:])
scores=pd.concat([pd.read_parquet(WORK/f'gpu{gpu}'/'scores.parquet') for gpu in range(2)],ignore_index=True)
ids=pd.read_parquet(pair_path,columns=['pair_id']).pair_id
assert not scores.pair_id.duplicated().any() and set(scores.pair_id)==set(ids)
assert set(scores.revision)=={manifest['revision']}
out=WORK/'scores.parquet'; scores.to_parquet(out,index=False)
run={'input':manifest,'score_sha256':sha(out),'statuses':scores.status.value_counts().to_dict(),
     'torch':torch.__version__,'truncation':False,'worker_sha256':sha(worker_path),
     'workers':[json.loads((WORK/f'gpu{gpu}'/'identity.json').read_text()) for gpu in range(2)]}
(WORK/'run.json').write_text(json.dumps(run,indent=2))
target=Path('/kaggle/working')/f'p2b_{SCOPE}_scores.zip'
with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(out,'scores.parquet'); z.write(WORK/'run.json','run.json')
print({'output':str(target),'sha256':sha(target),'statuses':run['statuses']})
